# Tutorial Assignment: Version Control, Collaboration, Git LFS, DVC, and Docker for MLOps

## Learning Objective
This tutorial assignment gives you hands-on practice with the core engineering tools required for reproducible MLOps workflows. You will create a small ML-style project, version the code using Git, isolate changes using branches, resolve a merge conflict, handle large model artifacts using Git LFS, version datasets using DVC, and containerize a small prediction service using Docker.

## Context
In normal software projects, code is usually the main artifact. In machine learning projects, the final result depends on five moving parts: code, data, configuration, environment, and randomness. A model can change even when the code is unchanged if the dataset, hyperparameters, dependency versions, or random seeds change. Therefore, an MLOps workflow must track not only source code, but also data, models, configs, and the execution environment.

By the end of this notebook, you should be able to explain and implement a minimal reproducible ML workflow using Git, GitHub-style collaboration, Git LFS, DVC, and Docker.

## Submission Instructions
Complete every task cell marked `TODO`. The solutions are provided at the bottom of this notebook for self-checking after completion.

## Setup Cell
Run this cell only if you are working in a fresh Python environment. Shell commands are written with `!` because this notebook is intended to be executed from Jupyter/Colab-style environments.

If Docker, Git LFS, or DVC are unavailable in your environment, still write the expected commands and explain what they would do.

In [ ]:
import os
import json
import textwrap
from pathlib import Path

PROJECT_DIR = Path("mlops_versioning_demo")
PROJECT_DIR.mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

# Part 1 — Create a Minimal ML Project

## Task 1: Build the Initial Project Structure

### Problem Description
You are starting a small ML project that predicts customer churn from a tiny CSV dataset. Before training any model, you need a clean project layout that separates source code, data, model outputs, and configuration. This separation is important because Git, DVC, Git LFS, and Docker will later track different parts of the project differently.

Create the following structure:

```text
mlops_versioning_demo/
├── src/
│   ├── prepare.py
│   └── train.py
├── data/
│   └── dataset.csv
├── models/
├── params.json
├── README.md
└── .gitignore
```

### Student Task
Write Python code to create the folders and files. The dataset should contain at least five rows with columns:

```text
customer_id,tenure_months,monthly_charge,contract,churn
```

The `.gitignore` should ignore Python cache files, virtual environments, raw CSV files, and model binaries.

In [ ]:
# TODO: Create src, data, and models directories.
# TODO: Create README.md, params.json, data/dataset.csv, src/prepare.py, src/train.py, and .gitignore.
# Hint: Use Path(...).mkdir(), Path(...).write_text(), and json.dumps(...).

# Part 2 — Git Basics: Snapshot, Diff, Commit, and History

## Task 2: Initialize Git and Create the First Commit

### Problem Description
Git is a distributed version control system. It tracks changes in text-based files such as Python scripts, configs, and documentation. The core local workflow is:

```text
working directory → staging area → local repository
```

You edit files in the working directory, select what should be saved using `git add`, and then create a snapshot using `git commit`.

### Student Task
Initialize a Git repository, inspect the status, stage all project files, commit them, and print the one-line commit history.

In [ ]:
# TODO: Run the Git commands below using shell syntax.
# 1. git init
# 2. git status
# 3. git add README.md params.json src .gitignore
# 4. git commit -m "Initial ML project structure"
# 5. git log --oneline

## Task 3: Modify a Hyperparameter and Inspect the Difference

### Problem Description
In ML, hyperparameters are part of the experiment definition. Changing the learning rate changes the model training behavior, so the change must be traceable. Before committing, use `git diff` to inspect exactly what changed.

### Student Task
Change the learning rate in `params.json` from `0.01` to `0.05`, inspect the diff, commit the change, and print the commit history.

In [ ]:
# TODO: Load params.json, change learning_rate to 0.05, and save the file.
# TODO: Run git status, git diff, git add params.json, git commit, and git log --oneline.

# Part 3 — Branching, Pull Requests, and Merge Conflicts

## Task 4: Create a Feature Branch for a README Update

### Problem Description
A branch is a movable pointer to a commit. Branches allow teams to isolate work. In GitHub Flow, developers usually create a short-lived branch, push it, open a pull request, review it, and merge it into `main`.

### Student Task
Create a branch called `fix/readme-description`, update the README to clarify that this is a hands-on MLOps demo, commit the change, switch back to `main`, merge the branch, and inspect the graph.

In [ ]:
# TODO: Create and switch to a new branch named fix/readme-description.
# TODO: Modify README.md.
# TODO: Commit the README change.
# TODO: Switch back to main or master depending on your repository default branch.
# TODO: Merge fix/readme-description.
# TODO: Display git log --oneline --graph --all.

## Task 5: Create and Resolve a Merge Conflict

### Problem Description
A merge conflict happens when two branches edit the same line differently. Git cannot decide which version is correct, so the developer must manually resolve the conflict. In ML projects, conflicts frequently occur in config files such as `params.json` when two experiments tune the same hyperparameter.

### Student Task
Create two experiment branches from the same base:

- `experiment/lr-high`: set `learning_rate = 0.1`
- `experiment/lr-low`: set `learning_rate = 0.001`

Merge `experiment/lr-high` into `main`, then merge `experiment/lr-low`. Resolve the conflict by choosing `learning_rate = 0.05`, commit the resolved file, and inspect the graph.

In [ ]:
# TODO: Create experiment/lr-high and set learning_rate to 0.1.
# TODO: Commit the change.
# TODO: Return to main/master.
# TODO: Create experiment/lr-low from the original base if possible and set learning_rate to 0.001.
# TODO: Commit the change.
# TODO: Merge lr-high, then merge lr-low.
# TODO: Resolve the conflict manually by keeping learning_rate = 0.05.
# TODO: Commit the merge resolution and show the graph.

# Part 4 — Git LFS for Large Model Files

## Task 6: Track Model Files with Git LFS

### Problem Description
Git stores every version of every committed file in history. Large binary files such as `.h5`, `.pt`, or `.pkl` model checkpoints can quickly bloat a repository. Git LFS solves this by storing a small text pointer in Git while storing the actual binary blob in an LFS store.

### Student Task
Configure Git LFS to track Keras model files (`*.h5`). Create a dummy model file in `models/model.h5`, add the `.gitattributes` file and model file, commit them, then verify which files are tracked by LFS.

In [ ]:
# TODO: Run git lfs install.
# TODO: Run git lfs track "*.h5".
# TODO: Create a dummy models/model.h5 file.
# TODO: Add .gitattributes and models/model.h5.
# TODO: Commit with message "Add trained model tracked by Git LFS".
# TODO: Run git lfs ls-files.

# Part 5 — DVC for Dataset and Pipeline Versioning

## Task 7: Initialize DVC and Track the Dataset

### Problem Description
Git LFS is useful for large files, but it does not model data lineage or ML pipelines. DVC is designed for versioning datasets, models, and reproducible pipelines. Git tracks small `.dvc` metadata files, while DVC stores actual data in its cache or remote storage.

### Student Task
Initialize DVC, track `data/dataset.csv`, commit the generated metadata, configure a local DVC remote, and push the data to the remote.

In [ ]:
# TODO: Run dvc init.
# TODO: Run dvc add data/dataset.csv.
# TODO: Add the generated .dvc file and .gitignore changes to Git.
# TODO: Commit with message "Track dataset v1 with DVC".
# TODO: Configure a local remote at /tmp/dvc-remote.
# TODO: Run dvc push.

## Task 8: Update the Dataset and Roll Back to Version 1

### Problem Description
Dataset versions change over time. A reproducible ML workflow must be able to recover older versions. With Git + DVC, Git checks out the metadata pointer and DVC checks out the corresponding data content.

### Student Task
Append two new rows to `data/dataset.csv`, run `dvc status`, re-add the dataset with DVC, commit the new `.dvc` metadata, push to DVC remote, and then roll back to the previous dataset version.

In [ ]:
# TODO: Append two new rows to data/dataset.csv.
# TODO: Run dvc status.
# TODO: Run dvc add data/dataset.csv.
# TODO: Commit the updated dataset metadata.
# TODO: Run dvc push.
# TODO: Use git checkout HEAD~1 data/dataset.csv.dvc and dvc checkout to restore version 1.
# TODO: Print data/dataset.csv to verify rollback.

## Task 9: Define a DVC Pipeline

### Problem Description
A DVC pipeline describes the process that transforms raw data into model artifacts. It stores the dependency graph in `dvc.yaml`, while `dvc.lock` records exact hashes of dependencies and outputs from the last successful run. When `dvc repro` is executed, only stages affected by changed inputs are re-run.

### Student Task
Create a two-stage pipeline:

1. `prepare`: runs `python src/prepare.py`, depends on `src/prepare.py` and `data/dataset.csv`, produces `data/prepared.csv`
2. `train`: runs `python src/train.py`, depends on `src/train.py`, `data/prepared.csv`, and `params.json`, produces `models/model.h5`

Run `dvc repro` and commit `dvc.yaml` and `dvc.lock`.

In [ ]:
# TODO: Update src/prepare.py so that it reads data/dataset.csv and writes data/prepared.csv.
# TODO: Update src/train.py so that it reads params.json and data/prepared.csv, then writes models/model.h5.
# TODO: Create DVC stages using dvc stage add.
# TODO: Run dvc repro.
# TODO: Commit dvc.yaml and dvc.lock.

# Part 6 — Docker for Reproducible Environments

## Task 10: Create a Minimal Prediction Service

### Problem Description
Containers provide lightweight isolation and portability. A Docker image contains application code, dependencies, and runtime instructions. A running container is an instance of an image. Docker helps ensure that the service runs consistently across machines.

You will create a tiny FastAPI app with two endpoints:

- `/health`: confirms that the service is running
- `/predict`: accepts a JSON payload containing texts and returns simple positive/negative labels

### Student Task
Create `app.py` and `requirements.txt` for a minimal prediction API.

In [ ]:
# TODO: Create app.py with FastAPI endpoints /health and /predict.
# TODO: Create requirements.txt with fastapi and uvicorn.

## Task 11: Write a Dockerfile and Run the Service

### Problem Description
A Dockerfile defines the image build process as a sequence of layers. Each instruction that changes the filesystem creates a new layer. A good Dockerfile keeps the image small, copies only required files, installs dependencies, and defines a clear command.

### Student Task
Write a Dockerfile that:

1. Uses `python:3.10-slim`
2. Sets `/app` as the working directory
3. Installs dependencies from `requirements.txt`
4. Copies `app.py`
5. Exposes port `8000`
6. Runs the app using Uvicorn

Then build and run the image.

In [ ]:
# TODO: Create Dockerfile.
# TODO: Build the image using docker build -t ml-app:v1 .
# TODO: Run the container using docker run -d --name ml-app-demo -p 8000:8000 ml-app:v1
# TODO: Test /health and /predict using curl.

## Task 12: Docker Debugging and Cleanup

### Problem Description
Containers and images consume disk space. In real projects, unused images, stopped containers, and build cache can grow quickly. Docker provides commands for inspecting containers, checking image sizes, viewing image layers, and pruning unused resources.

### Student Task
Run commands to inspect the container, list image sizes, view image history, stop/remove the container, and remove unused Docker resources.

In [ ]:
# TODO: Run docker ps.
# TODO: Run docker inspect ml-app-demo.
# TODO: Run docker images --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}".
# TODO: Run docker history ml-app:v1.
# TODO: Stop and remove the container.
# TODO: Optionally run docker system prune.

# Part 7 — Conceptual Questions

Answer these briefly in your own words.

1. Why is code versioning alone insufficient for ML reproducibility?
2. What is the difference between Git LFS and DVC?
3. Why should raw datasets and model binaries usually not be committed directly to Git?
4. Why is one hypothesis per branch useful in ML experiments?
5. What is the difference between a Docker image and a Docker container?
6. How do namespaces and cgroups help containers provide isolation?
7. Why does Docker Compose become useful when an application has multiple services?

In [ ]:
# TODO: Write your answers in Markdown below this cell.

---

# Solutions Appendix

The following cells provide one possible solution. Try the tasks before reading this section.

## Solution 1: Project Structure

In [ ]:
from pathlib import Path
import json

Path("src").mkdir(exist_ok=True)
Path("data").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

Path("README.md").write_text("# Tiny MLOps Demo\n\nThis project demonstrates version control, data versioning, and containerized serving.\n")
Path("params.json").write_text(json.dumps({"learning_rate": 0.01, "epochs": 5, "seed": 42}, indent=2))
Path("data/dataset.csv").write_text("customer_id,tenure_months,monthly_charge,contract,churn\n1,12,70.5,month-to-month,1\n2,24,55.0,one-year,0\n3,5,88.2,month-to-month,1\n4,48,40.0,two-year,0\n5,9,77.5,month-to-month,1\n")
Path("src/prepare.py").write_text("from pathlib import Path\nimport pandas as pd\n\ndf = pd.read_csv('data/dataset.csv')\ndf['is_month_to_month'] = (df['contract'] == 'month-to-month').astype(int)\nPath('data').mkdir(exist_ok=True)\ndf.to_csv('data/prepared.csv', index=False)\nprint('prepared rows:', len(df))\n")
Path("src/train.py").write_text("from pathlib import Path\nimport json\nimport pandas as pd\n\nparams = json.loads(Path('params.json').read_text())\ndf = pd.read_csv('data/prepared.csv') if Path('data/prepared.csv').exists() else pd.read_csv('data/dataset.csv')\nPath('models').mkdir(exist_ok=True)\nPath('models/model.h5').write_text(f'dummy model trained with lr={params[\"learning_rate\"]}, rows={len(df)}\\n')\nprint('model written to models/model.h5')\n")
Path(".gitignore").write_text("__pycache__/\n.venv/\n.env\ndata/*.csv\nmodels/*.h5\n")
print("Project files created.")

## Solution 2: Git Initialization and First Commit

In [ ]:
# Run in a shell or notebook cell:
# !git init
# !git status
# !git add README.md params.json src .gitignore
# !git commit -m "Initial ML project structure"
# !git log --oneline

## Solution 3: Modify Hyperparameter and Commit

In [ ]:
import json
from pathlib import Path

params = json.loads(Path("params.json").read_text())
params["learning_rate"] = 0.05
Path("params.json").write_text(json.dumps(params, indent=2))

# !git status
# !git diff
# !git add params.json
# !git commit -m "Tune learning rate to 0.05"
# !git log --oneline

## Solution 4: Branch and Merge

In [ ]:
# !git switch -c fix/readme-description
# Update README.md manually or by Python:
from pathlib import Path
Path("README.md").write_text("# Tiny, Hands-on MLOps Demo\n\nThis project demonstrates Git, DVC, Git LFS, and Docker for reproducible ML workflows.\n")
# !git add README.md
# !git commit -m "Clarify project description in README"
# !git switch main || git switch master
# !git merge fix/readme-description
# !git log --oneline --graph --all

## Solution 5: Merge Conflict

In [ ]:
# One clean way is to start both branches from the same earlier commit.
# Commands may vary depending on whether your default branch is main or master.

# !git switch main
# !git switch -c experiment/lr-high
# Change params.json learning_rate to 0.1, then:
# !git add params.json
# !git commit -m "Try LR = 0.1"

# !git switch main
# !git switch -c experiment/lr-low HEAD~1
# Change params.json learning_rate to 0.001, then:
# !git add params.json
# !git commit -m "Try LR = 0.001"

# !git switch main
# !git merge experiment/lr-high
# !git merge experiment/lr-low

# After conflict appears, edit params.json and keep:
# {
#   "learning_rate": 0.05,
#   "epochs": 5,
#   "seed": 42
# }

# !git add params.json
# !git commit --no-edit
# !git log --oneline --graph --all

## Solution 6: Git LFS

In [ ]:
# !git lfs install
# !git lfs track "*.h5"
from pathlib import Path
Path("models").mkdir(exist_ok=True)
Path("models/model.h5").write_bytes(b"0" * 1024 * 1024)
# !git add .gitattributes models/model.h5
# !git commit -m "Add trained model tracked by Git LFS"
# !git lfs ls-files
# !git show HEAD:models/model.h5

## Solution 7: DVC Dataset Tracking

In [ ]:
# !pip install dvc
# !dvc init
# !dvc add data/dataset.csv
# !git add data/dataset.csv.dvc data/.gitignore .dvc .dvcignore
# !git commit -m "Track dataset v1 with DVC"
# !mkdir -p /tmp/dvc-remote
# !dvc remote add -d localstore /tmp/dvc-remote
# !git add .dvc/config
# !git commit -m "Configure DVC remote"
# !dvc push

## Solution 8: Dataset Update and Rollback

In [ ]:
from pathlib import Path
with Path("data/dataset.csv").open("a") as f:
    f.write("6,60,99.10,two-year,0\n")
    f.write("7,1,45.20,month-to-month,1\n")

# !dvc status
# !dvc add data/dataset.csv
# !git add data/dataset.csv.dvc
# !git commit -m "Update dataset to v2"
# !dvc push
# !git checkout HEAD~1 data/dataset.csv.dvc
# !dvc checkout
# !cat data/dataset.csv

## Solution 9: DVC Pipeline

In [ ]:
# Ensure scripts exist first.
from pathlib import Path
Path("src/prepare.py").write_text("from pathlib import Path\nimport pandas as pd\n\ndf = pd.read_csv('data/dataset.csv')\ndf['is_month_to_month'] = (df['contract'] == 'month-to-month').astype(int)\nPath('data').mkdir(exist_ok=True)\ndf.to_csv('data/prepared.csv', index=False)\nprint('prepared rows:', len(df))\n")
Path("src/train.py").write_text("from pathlib import Path\nimport json\nimport pandas as pd\n\nparams = json.loads(Path('params.json').read_text())\ndf = pd.read_csv('data/prepared.csv')\nPath('models').mkdir(exist_ok=True)\nPath('models/model.h5').write_text(f'dummy model trained with lr={params[\"learning_rate\"]}, rows={len(df)}\\n')\nprint('model written to models/model.h5')\n")

# !dvc stage add -n prepare -d src/prepare.py -d data/dataset.csv -o data/prepared.csv python src/prepare.py
# !dvc stage add -n train -d src/train.py -d data/prepared.csv -d params.json -o models/model.h5 python src/train.py
# !dvc repro
# !git add dvc.yaml dvc.lock src/prepare.py src/train.py
# !git commit -m "Add reproducible DVC pipeline"

## Solution 10: FastAPI Prediction Service

In [ ]:
from pathlib import Path
Path("app.py").write_text('''from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="MLOps Demo API")

class PredictRequest(BaseModel):
    texts: list[str]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(request: PredictRequest):
    predictions = []
    positive_words = {"love", "great", "excellent", "good", "happy"}
    for text in request.texts:
        label = "POSITIVE" if any(word in text.lower() for word in positive_words) else "NEGATIVE"
        predictions.append({"text": text, "label": label})
    return {"predictions": predictions}
''')
Path("requirements.txt").write_text("fastapi\nuvicorn[standard]\npydantic\n")
print("API files created.")

## Solution 11: Dockerfile and Run Commands

In [ ]:
from pathlib import Path
Path("Dockerfile").write_text('''FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
''')

# !docker build -t ml-app:v1 .
# !docker run -d --name ml-app-demo -p 8000:8000 ml-app:v1
# !curl http://localhost:8000/health
# !curl -X POST http://localhost:8000/predict -H "Content-Type: application/json" -d '{"texts": ["I love this!", "This is awful."]}'

## Solution 12: Docker Debugging and Cleanup

In [ ]:
# !docker ps
# !docker inspect ml-app-demo
# !docker images --format "table {{.Repository}}\t{{.Tag}}\t{{.Size}}"
# !docker history ml-app:v1
# !docker stop ml-app-demo
# !docker rm ml-app-demo
# !docker system prune -f

## Conceptual Answer Key

1. Code versioning alone is insufficient because ML outputs depend on code, data, configuration, environment, and randomness.
2. Git LFS stores large files through Git pointer files, while DVC versions data, models, remotes, and pipelines with lineage.
3. Raw datasets and model binaries should not be committed directly because they are large, do not produce meaningful text diffs, and permanently bloat repository history.
4. One hypothesis per branch makes metric changes attributable to a single cause, which improves experiment comparison.
5. A Docker image is a read-only build artifact; a Docker container is a running instance of that image with a writable layer.
6. Namespaces isolate views of resources such as filesystem and network; cgroups limit CPU, memory, and other resource usage.
7. Docker Compose is useful when an application has multiple services because it defines and starts them together on a shared network.